#### Metadata Ingestion Framework

This metadata-driven ingestion code is designed for efficient data processing, focusing on transitioning data from the Landing to Bronze layers and Bronze to Silver Staging layers.

The primary objectives of this code are:
1. Read streaming data from the layers.
2. Add additional metadata such as batch year, month, day, batch ID and file name.
3. Flatten nested JSON structures to make the data more accessible and query-friendly.
4. Validate the schema of the incoming data against a predefined schema stored in a text file.
5. Write the processed and validated data to the Silver Staging layer as Delta tables, partitioned by batch metadata.

The processing steps include defining parameters, importing necessary packages, defining functions for schema validation, reading configurations from config file, extracting file metadata, adding new columns, flattening JSON data, and finally, writing the processed data to Delta tables.

Note: Ensure to adjust parameters such as source and destination storage account details, database name, table name, etc., according to your specific environment.

In [59]:
# Import necessary pakages
from pyspark.sql.types import StructType, ArrayType, StringType, TimestampType, DataType, StructField, LongType
from pyspark.sql.functions import col, explode, input_file_name, split, lit, current_timestamp, array, struct
from notebookutils import mssparkutils
import time

Define variables crucial for interacting with a storage account and configuring a data processing task. It specifies the name of the storage account, the container where configuration files are stored, error table path, and the relative path to a specific JSON configuration file. Additionally, it sets a stage parameter indicating a phase in a data processing pipeline.

In [60]:
# Storage Account
storage_account_name = "stgipcemeapoc"

# Config Parameters
mdif_config_container = "config"
mdif_config_relative_path = "purchase_orders/purchase_orders_config.json"
mdif_schema_relative_path = "purchase_orders/purchase_orders_schema.txt"
mdif_bronze_stage = "landing_to_bronze"
mdif_silver_stage = "bronze_to_silver_staging"

# Error Table Parameters
error_db_name = "silver"
error_table_name = "error_table_mdif"
error_table_container = "silver"

# Batch Id
batch_id = "20240519163842"

###### Define Functions

The transform_df function serves as a versatile tool for transforming DataFrames based on specified rules. It iterates through a list of transformations, sorting them based on a specified order, and applies actions such as column renaming or adding new columns accordingly. This function offers flexibility in handling diverse transformation requirements, enhancing the adaptability of the data processing pipeline.

In [61]:
# Function to transform dataframe
def transform_df(df, transformations):
    # Sort transformations based on order
    transformations.sort(key=lambda x: x["order"])
    
    for transformation in transformations:
        transformation_action = transformation.get("action")
        column_name = transformation.get("column_name")

        if transformation_action == "rename_column":
            alias = transformation.get("alias")
            df = df.withColumnRenamed(column_name, alias)

        elif transformation_action == "add_column":
            tranformation_type = transformation.get("tranformation_type")
            if tranformation_type == "eval":
                transformation_expr = transformation.get("transformation_expression")
                df = df.withColumn(column_name, eval(transformation_expr))

    df = df.withColumn("batch_id", lit(batch_id))
    
    return df

The flatten_df function addresses the challenge of dealing with nested structures within DataFrame schemas. By recursively traversing the schema and constructing flattened field names, this function effectively simplifies complex structures, making the data more accessible for downstream processing tasks. It offers a streamlined approach to handling nested data, promoting clarity and ease of use within the pipeline.

In [62]:
# Function to flatten dataframe
def flatten_df(df, prefix_remove=None):
    def flatten(schema, prefix=None, prefix_remove=None):
        fields = []
        for field in schema.fields:
            name = prefix + '.' + field.name if prefix else field.name
            dtype = field.dataType
            
            if isinstance(dtype, StructType):
                fields += flatten(dtype, prefix=name, prefix_remove=prefix_remove)
            else:
                fields.append(name)

        return fields
    
    # Get actual field names, with nested '.' structure, and create equivalents with '_'
    fields = flatten(df.schema, prefix_remove=prefix_remove)
    fields_renamed = []

    for field in fields:
        if prefix_remove and field.startswith(prefix_remove):
            field = field[len(prefix_remove):]
            
        fields_renamed.append(field.replace(".","__"))

    # Select while aliasing for all fields
    df = df.select(*[col(field).alias(new_field) for field, new_field in zip(fields, fields_renamed)])

    return df

The explode_columns function targets columns with nested structures, providing functionality to explode array-type columns while also flattening struct-type columns. This capability is vital for scenarios where data expansion is necessary to access individual elements within arrays or structs. By enabling this operation, the function facilitates granular data manipulation and analysis, enhancing the pipeline's capabilities in handling intricate data structures.

In [63]:
 # Function to explode columns in dataframe
def explode_columns(df, columns_to_explode=None):
    for column in columns_to_explode:
        if isinstance(df.schema[column].dataType, ArrayType):
            df = df.withColumn(column, explode(col(column)))
        
        if isinstance(df.schema[column].dataType, StructType):
            df = flatten_df(df)

    return df

The rename_columns function takes a DataFrame (df) and a dictionary (col_map) that maps source column names to target column names. It returns a new DataFrame where the columns specified in the col_map are renamed accordingly. The resulting DataFrame contains the original data with the specified columns renamed as per the provided mapping.

In [64]:
# Function to rename columns based on the dictionary
def rename_columns(df, col_map):
    return df.select([col(source).alias(target) for source, target in col_map.items()])

The find_fields_of_json function recursively explores a given JSON schema represented by a DataType object. It systematically traverses through the schema, identifying fields and their paths, and appends these paths to a provided list. If encountering nested structures (StructType), it iterates through each field, appending their names to the current path. Similarly, if encountering arrays (ArrayType), it delves into their element types. Ultimately, it collects paths to all fields, regardless of their depth or presence within arrays, offering a comprehensive overview of the schema's structure.

In [65]:
# Function for finding all fields of JSON
def find_fields_of_json(path: str, dt: DataType, lstschema: []):
    if isinstance(dt, StructType):
        for f in dt.fields:
            find_fields_of_json(path + "." + f.name, f.dataType, lstschema)
    elif isinstance(dt, ArrayType):
        find_fields_of_json(path, dt.elementType, lstschema)
    else:
        lstschema.append(path)

The validate_schema funtion compares the schema of a DataFrame against a predefined schema. This function detects any disparities, such as missing or additional attributes, and reports them for further investigation. By ensuring schema conformity, it safeguards the consistency and quality of the processed data, mitigating potential errors or discrepancies in downstream operations.

In [66]:
# Function to validate schema and report additional columns
def validate_schema(df):
    lst_batch_schema = []
    lst_schema = []
    error_dfs = []

    schema_path = f"abfss://{mdif_config_container}@{storage_account_name}.dfs.core.windows.net/{mdif_schema_relative_path}"
    schema_df = spark.read.text(schema_path, wholetext=True)
    schema_df = spark.createDataFrame([], schema=eval(schema_df.first()['value']))
    df = df.withColumn("data", explode(array("data"))).select("data.*", "batch_year", "batch_month", "batch_day", "batch_id", "file_name")

    find_fields_of_json("", df.schema, lst_batch_schema)    
    find_fields_of_json("", schema_df.schema, lst_schema)

    # Finding additional and missing attributes in the batch
    unknown_columns = list(set(lst_schema) ^ set(lst_batch_schema))

    # If unknown_columns is empty, the schema is validated
    if unknown_columns != []:
        for column in unknown_columns:

            # Report additional attribute in the error table
            if (column in lst_batch_schema) and (column not in lst_schema) and (column.count(".") == 1):
                column = column.split(".")[-1]
                
                error_df = df.filter(df[column].isNotNull()) \
                                .withColumn("error_column", lit(column)) \
                                .withColumn("error_code", lit(1002)) \
                                .withColumn("error_description", lit("Schema Validation: Received additional attribute")) \
                                .withColumn("created_ts", current_timestamp()) \
                                .withColumn("updated_ts", lit(None).cast(TimestampType())) \
                                .select(col("error_column"), col("error_code"), col("error_description"), col("batch_id"), col("file_name"), col(column).cast("string").alias("error_data"), col("created_ts"), col("updated_ts"))

                error_dfs.append(error_df)

            # Add missing attribute
            elif (column not in lst_batch_schema) and (column in lst_schema) and (column.count(".") == 1):
                column = column.split(".")[-1]
                df = df.withColumn(column, lit(None))
                
    return df, error_dfs

The create_error_table function helps in error handling and logging within the pipeline. By establishing an error table with specified attributes, this function provides a structured mechanism for capturing and storing error-related information. It enhances the pipeline's robustness by enabling systematic error tracking and analysis, facilitating proactive measures to address and mitigate potential issues.

In [67]:
# Function for creating error table if not exists
def create_error_table():
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {error_db_name}.{error_table_name}
              (
                  error_column string,
                  error_code int,
                  error_description string,
                  batch_id string,
                  file_name string,
                  error_data string,
                  created_ts timestamp,
                  updated_ts timestamp
              )
              USING DELTA
              LOCATION "abfss://{error_table_container}@{storage_account_name}.dfs.core.windows.net/{error_table_name}/";""")
    time.sleep(10)

The write_df_to_target function encapsulates the functionality for writing DataFrame output to designated target locations. With configurable options for format, path, checkpoint location, and partitioning, this function offers flexibility in tailoring the output to diverse requirements. By orchestrating the data writing process, it ensures seamless integration with target systems or storage platforms, enabling efficient data delivery within the pipeline.

In [68]:
# Function to write dataframe to target
def write_df_to_target(df, format, path, db, table, mode, partition_keys):
    df.writeStream \
        .format(format) \
        .option("mergeSchema", "true") \
        .option("path", path) \
        .option("checkpointLocation",f"{path}_checkpoint") \
        .outputMode(mode) \
        .partitionBy(partition_keys) \
        .trigger(availableNow=True) \
        .toTable(f"{db}.{table}")
    time.sleep(10)

The run_mdif function reads source data into a DataFrame based on configuration settings, handling both streaming and static sources. If schema inference is enabled, it configures Spark for automatic schema detection. The data is then read into a DataFrame and transformed using the transform_df function if specified in the configuration. This ensures the data is processed correctly before being written to the target. For target processing, the function iterates through each target configuration, constructs the target path, and writes the transformed DataFrame using the write_df_to_target function. This process ensures efficient and accurate data transfer from source to target, leveraging partitioning and schema merging for optimized large-scale, real-time data processing.

In [69]:
def run_mdif(config, stage):    
    # Read source and target configs
    source_config_key = "source"
    target_config_key = "target"

    stage_config = config_df.select(stage).first()[0].asDict(recursive=True)
    source_config = stage_config.get(source_config_key)
    target_config = stage_config.get(target_config_key)

    # Read source data into dataframe
    source_data_layer = source_config.get("data_layer")
    source_storage_account = source_config.get("storage_account")
    source_container = source_config.get("container")
    source_relative_path = source_config.get("relative_path")
    source_format = source_config.get("format")
    source_read_as = source_config.get("read_as")
    source_infer_schema = source_config.get("infer_schema")
    source_type = source_config.get("type")
    source_multiline = source_config.get("multiline")
    source_transformations = source_config.get("transformations", [])
    source_path = f"abfss://{source_container}@{source_storage_account}.dfs.core.windows.net/{source_relative_path}/"

    if source_infer_schema == "infer":
        # Allow automatic schema inference while reading
        spark.conf.set("spark.sql.streaming.schemaInference", True)

    # Read data into streaming dataframe
    if source_read_as == "stream":
        
        # Read source data as file
        if source_type == "file":
            source_df = spark.readStream \
                .format(source_format) \
                .option("multiLine", True) \
                .load(source_path)

        # Read source data as table
        else:
            source_df = spark.readStream \
                .format(source_format) \
                .load(source_path)

    # Read data into static dataframe    
    else:
        source_df = spark.read \
            .format(source_format) \
            .load(source_path)
    
    if stage == "landing_to_bronze":    
        source_df = source_df.select(struct(*source_df.columns).alias("value"))

    if source_transformations:
        source_df = transform_df(source_df, source_transformations)

    # Target processing
    target_data_layer = target_config.get("data_layer")
    targets = target_config.get("targets")

    for target in targets:
        target_storage_account = target.get("storage_account")
        flatten_data = target.get("flatten")
        validate_data = target.get("validate_schema")
        prefix_remove = target.get("prefix_remove")
        columns_to_explode = target.get("explode_columns")
        source_to_target_mapping = target.get("source_to_target_mapping")
        target_container = target.get("container")
        target_relative_path = target.get("relative_path")
        target_db = target.get("db")
        target_table = target.get("table")
        target_format = target.get("format")
        output_mode = target.get("mode")
        partition_keys = target.get("partition_cols", [])

        target_path = f"abfss://{target_container}@{target_storage_account}.dfs.core.windows.net/{target_relative_path}/"

        target_df = source_df

        if validate_data == True:
            target_df, error_dfs = validate_schema(target_df)
            if error_dfs == []:
                create_error_table()
            else:
                for error_df in error_dfs:
                    error_df.writeStream \
                            .format("delta") \
                            .option("path", f"abfss://{error_table_container}@{storage_account_name}.dfs.core.windows.net/{error_table_name}/") \
                            .option("checkpointLocation",f"abfss://{error_table_container}@{storage_account_name}.dfs.core.windows.net/{error_table_name}/_checkpoint") \
                            .outputMode("append") \
                            .trigger(availableNow=True) \
                            .toTable(f"{error_db_name}.{error_table_name}")

        if flatten_data == True:
            target_df = flatten_df(target_df, prefix_remove=prefix_remove)

        if columns_to_explode:
            target_df = explode_columns(target_df, columns_to_explode)

        target_df = rename_columns(target_df, source_to_target_mapping)

        write_df_to_target(target_df, target_format, target_path, target_db, target_table, output_mode, partition_keys)

    time.sleep(20)

In [70]:
# Read the config file for source and target
config_source_path = f"abfss://{mdif_config_container}@{storage_account_name}.dfs.core.windows.net/{mdif_config_relative_path}"
config_df = spark.read.json(config_source_path, multiLine=True)

In [71]:
# Run mdif for ingesting data from landing to bronze
run_mdif(config_df, mdif_bronze_stage)

In [72]:
# Run mdif for ingesting data from bronze to silver staging
run_mdif(config_df, mdif_silver_stage)